# Question 7 - Permian Delaware Tight cohort datasets

This notebook completes Question 7 of the assignment. It opens every available Permian Delaware Tight exposure dataset for both spatial resolutions (regions and countries), displays each cohort's dimensions and variables, and combines each group into one xarray Dataset with a `birth_years` coordinate.

The supplied files contain 39 cohorts: 1950–1968, 1990, and 2007–2025. Missing years are not invented or interpolated. The regional data has 9 regions and the country data has 183 countries; both cover 6 climate extremes. The merged datasets are saved as `Data/question7_regions_by_birth_year.pkl` and `Data/question7_countries_by_birth_year.pkl`.

Question 8, the regional exposure map, will be handled in its own question-numbered file after this Question 7 stage is reviewed and committed.

## Step 1 - Find the project data folder and load the Question 7 functions

In [2]:
from pathlib import Path
import sys

project_root = next(
    (
        folder
        for folder in (Path.cwd(), *Path.cwd().parents)
        if (folder / "Data").is_dir()
    ),
    None,
)
if project_root is None:
    raise FileNotFoundError("Could not find the project's Data folder.")
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

from scripts.question7_cohort_datasets import (
    load_cohort_datasets,
    merge_cohort_datasets,
)

data_dir = project_root / "Data"

## Step 2 - Load and inspect all available cohorts

The file names identify each cohort's birth year. We load the regional and country files separately, then print a compact description of every dataset so the notebook remains readable while still showing its structure.

In [3]:
regional_cohorts = load_cohort_datasets(data_dir, "regions")
country_cohorts = load_cohort_datasets(data_dir, "countries")

for label, cohort_datasets in [
    ("Regions", regional_cohorts),
    ("Countries", country_cohorts),
]:
    print(f"\n{label}: {len(cohort_datasets)} available cohorts")
    for birth_year, cohort_dataset in cohort_datasets.items():
        print(
            f"{birth_year}: dimensions={dict(cohort_dataset.sizes)}, "
            f"variables={list(cohort_dataset.data_vars)}"
        )

print("\nExample regional dataset for 2025:")
print(regional_cohorts[2025])
print("\nExample country dataset for 2025:")
print(country_cohorts[2025])


Regions: 39 available cohorts
1950: dimensions={'project': 1, 'extreme': 6, 'region': 9, 'samples': 1}, variables=['mean_val', 'std_val', 'median_val', 'ci_lower', 'ci_upper', 'emissions_best_estimate', 'emissions_std', 'tcre_best_estimate', 'tcre_std', 'tcre_a', 'tcre_b', 'n_samples', 'ic', 'seed_used']
1951: dimensions={'project': 1, 'extreme': 6, 'region': 9, 'samples': 1}, variables=['mean_val', 'std_val', 'median_val', 'ci_lower', 'ci_upper', 'emissions_best_estimate', 'emissions_std', 'tcre_best_estimate', 'tcre_std', 'tcre_a', 'tcre_b', 'n_samples', 'ic', 'seed_used']
1952: dimensions={'project': 1, 'extreme': 6, 'region': 9, 'samples': 1}, variables=['mean_val', 'std_val', 'median_val', 'ci_lower', 'ci_upper', 'emissions_best_estimate', 'emissions_std', 'tcre_best_estimate', 'tcre_std', 'tcre_a', 'tcre_b', 'n_samples', 'ic', 'seed_used']
1953: dimensions={'project': 1, 'extreme': 6, 'region': 9, 'samples': 1}, variables=['mean_val', 'std_val', 'median_val', 'ci_lower', 'ci_upp

## Step 3 - Merge each spatial resolution along `birth_years`

Before combining, the helper checks that each file has the expected resolution coordinate and that cohort dimensions and coordinate labels agree. `xr.concat` then adds `birth_years` as the cohort axis while preserving the original project, hazard, and spatial coordinates.

In [4]:
regional_by_birth_year = merge_cohort_datasets(regional_cohorts, "regions")
countries_by_birth_year = merge_cohort_datasets(country_cohorts, "countries")

print("Merged regional dataset:")
print(regional_by_birth_year)
print("\nMerged country dataset:")
print(countries_by_birth_year)

assert "birth_years" in regional_by_birth_year.coords
assert "birth_years" in countries_by_birth_year.coords
assert regional_by_birth_year.sizes["birth_years"] == len(regional_cohorts)
assert countries_by_birth_year.sizes["birth_years"] == len(country_cohorts)

Merged regional dataset:
<xarray.Dataset> Size: 236kB
Dimensions:                  (birth_years: 39, project: 1, extreme: 6,
                              region: 9, samples: 1)
Coordinates:
  * birth_years              (birth_years) int64 312B 1950 1951 ... 2024 2025
  * project                  (project) object 8B 'Permian Delaware Tight'
  * extreme                  (extreme) object 48B 'burntarea' ... 'tropicalcy...
  * region                   (region) object 72B 'Africa' 'Asia' ... 'World'
  * samples                  (samples) int64 8B 0
Data variables: (12/14)
    mean_val                 (birth_years, project, extreme, region) float64 17kB ...
    std_val                  (birth_years, project, extreme, region) float64 17kB ...
    median_val               (birth_years, project, extreme, region) float64 17kB ...
    ci_lower                 (birth_years, project, extreme, region) float64 17kB ...
    ci_upper                 (birth_years, project, extreme, region) float64 17kB

## Step 4 - Save the merged datasets

Saving these merged datasets makes the output of Question 7 reusable in later notebook sections, including the regional map in Question 8.

In [5]:
import pickle

merged_paths = {
    "regions": data_dir / "question7_regions_by_birth_year.pkl",
    "countries": data_dir / "question7_countries_by_birth_year.pkl",
}
for merged_path, merged_dataset in [
    (merged_paths["regions"], regional_by_birth_year),
    (merged_paths["countries"], countries_by_birth_year),
]:
    with merged_path.open("wb") as output_file:
        pickle.dump(merged_dataset, output_file, protocol=pickle.HIGHEST_PROTOCOL)
    print(f"Saved: {merged_path}")

Saved: c:\Users\USER\OneDrive\Desktop\Lune\CARBON-BOMB-PINGPONG-GROUP\Data\question7_regions_by_birth_year.pkl
Saved: c:\Users\USER\OneDrive\Desktop\Lune\CARBON-BOMB-PINGPONG-GROUP\Data\question7_countries_by_birth_year.pkl
